In [213]:
import setuptools
import matplotlib.pyplot as plt
import pandas as pd
from sklearn import (
    ensemble,
    preprocessing,
    tree,
)

from sklearn.metrics import (
    auc,
    roc_auc_score,
    confusion_matrix,
    roc_curve,
)

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
)

from yellowbrick.classifier import (
    ConfusionMatrix,
    ROCAUC,
)

from yellowbrick.model_selection import (
    LearningCurve,
)   



In [214]:
df_sujo = pd.read_csv("../data/raw/train.csv")
df_testes = pd.read_csv("../data/raw/test.csv")

df_sujo.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [215]:
from ydata_profiling import ProfileReport

profile = ProfileReport(df_sujo, title="Pandas Profiling Report") 
profile.to_file('../data/reports/relatorio_sujo.html')

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 50.46it/s]


In [216]:
df_sujo.dtypes.value_counts()

int64      5
object     5
float64    2
Name: count, dtype: int64

In [217]:
df_sujo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [218]:
df_sujo.isnull().sum().sort_values(ascending=False)

Cabin          687
Age            177
Embarked         2
PassengerId      0
Name             0
Pclass           0
Survived         0
Sex              0
Parch            0
SibSp            0
Fare             0
Ticket           0
dtype: int64

In [219]:
df_testes.isnull().sum().sort_values(ascending=False)

Cabin          327
Age             86
Fare             1
Name             0
Pclass           0
PassengerId      0
Sex              0
Parch            0
SibSp            0
Ticket           0
Embarked         0
dtype: int64

In [220]:
df = df_sujo.drop(columns=[ 'Name', 'Ticket'], axis=1)
df_testes = df_testes.drop(columns=[ 'Cabin', 'Name', 'Ticket'], axis=1)


In [221]:
df.columns = df.columns.str.lower()
df_testes.columns = df_testes.columns.str.lower()

In [222]:
df.loc[df['age'].isnull(), 'age'] = df['age'].mean()
df_testes.loc[df_testes['age'].isnull(), 'age'] = df_testes['age'].mean()

In [223]:
df.embarked.value_counts()

embarked
S    644
C    168
Q     77
Name: count, dtype: int64

In [224]:
df_testes.embarked.value_counts()


embarked
S    270
C    102
Q     46
Name: count, dtype: int64

In [225]:
df.loc[df['embarked'].isnull(), 'embarked'] = df['embarked'].mode()[0]
df_testes.loc[df_testes['embarked'].isnull(), 'embarked'] = df_testes['embarked'].mode()[0]

In [226]:
df.loc[df["fare"].isnull(), "fare"] 

Series([], Name: fare, dtype: float64)

In [227]:
df_testes.loc[df_testes["fare"].isnull(), "fare"]  = df_testes["fare"].mean()
df.loc[df["fare"].isnull(), "fare"]  = df["fare"].mean()

In [228]:
df_testes.isnull().sum().sort_values(ascending=False)


passengerid    0
pclass         0
sex            0
age            0
sibsp          0
parch          0
fare           0
embarked       0
dtype: int64

In [229]:
df.isnull().sum().sort_values(ascending=False)

cabin          687
passengerid      0
survived         0
pclass           0
age              0
sex              0
sibsp            0
parch            0
fare             0
embarked         0
dtype: int64

In [230]:
df_testes.isnull().sum().sort_values(ascending=False)

passengerid    0
pclass         0
sex            0
age            0
sibsp          0
parch          0
fare           0
embarked       0
dtype: int64

In [231]:
df.to_csv("../data/interim/titanic_limpo.csv", index=False)
df_testes.to_csv("../data/interim/titanic_testes_limpo.csv", index=False)

In [232]:
treino_nr = df[df.columns[df.dtypes != 'object']]

In [233]:
treino_nr

,passengerid,survived,pclass,age,sibsp,parch,fare
0,1,0,3,22.000000,1,0,7.2500
1,2,1,1,38.000000,1,0,71.2833
2,3,1,3,26.000000,0,0,7.9250
3,4,1,1,35.000000,1,0,53.1000
4,5,0,3,35.000000,0,0,8.0500
...,...,...,...,...,...,...,...
886,887,0,2,27.000000,0,0,13.0000
887,888,1,1,19.000000,0,0,30.0000
888,889,0,3,29.699118,1,2,23.4500
889,890,1,1,26.000000,0,0,30.0000


In [234]:

testes_nr = df_testes[df_testes.columns[df_testes.dtypes != 'object']]

In [235]:
testes_nr

,passengerid,pclass,age,sibsp,parch,fare
0,892,3,34.50000,0,0,7.8292
1,893,3,47.00000,1,0,7.0000
2,894,2,62.00000,0,0,9.6875
3,895,3,27.00000,0,0,8.6625
4,896,3,22.00000,1,1,12.2875
...,...,...,...,...,...,...
413,1305,3,30.27259,0,0,8.0500
414,1306,1,39.00000,0,0,108.9000
415,1307,3,38.50000,0,0,7.2500
416,1308,3,30.27259,0,0,8.0500


In [236]:
treino_nr.to_csv("../data/interim/titanic_numerical.csv", index=False)
testes_nr.to_csv("../data/interim/titanic_testes_numerical.csv", index=False)